## Example 1: Basic Array Type Annotations

Basic usage of jaxtyping for array shape and dtype annotations.

In [ ]:
# Example 1: Basic array type annotations
from jaxtyping import Float, Int, Bool
import numpy as np
from typing import Union

try:
    print("=== Basic Type Annotations ===")
    
    # Define functions with jaxtyping annotations
    def matrix_multiply(
        a: Float[np.ndarray, "m n"],
        b: Float[np.ndarray, "n k"]
    ) -> Float[np.ndarray, "m k"]:
        """Matrix multiplication with shape type hints"""
        return np.matmul(a, b)
    
    def compute_mean(
        data: Float[np.ndarray, "batch_size features"]
    ) -> Float[np.ndarray, "features"]:
        """Compute mean across batch dimension"""
        return np.mean(data, axis=0)
    
    def index_select(
        array: Float[np.ndarray, "n"],
        indices: Int[np.ndarray, "m"]
    ) -> Float[np.ndarray, "m"]:
        """Select elements by indices"""
        return array[indices]
    
    print("Functions defined with jaxtyping annotations")
    
    print("\n=== Testing Matrix Multiplication ===")
    # Test matrix multiplication
    A = np.random.randn(3, 4).astype(np.float32)
    B = np.random.randn(4, 5).astype(np.float32)
    
    print(f"Matrix A shape: {A.shape}, dtype: {A.dtype}")
    print(f"Matrix B shape: {B.shape}, dtype: {B.dtype}")
    
    C = matrix_multiply(A, B)
    print(f"Result C shape: {C.shape}, dtype: {C.dtype}")
    print(f"Expected shape: (3, 5) - Match: {C.shape == (3, 5)}")
    
    print("\n=== Testing Batch Mean Computation ===")
    batch_data = np.random.randn(100, 10).astype(np.float32)
    print(f"Batch data shape: {batch_data.shape}")
    
    mean_result = compute_mean(batch_data)
    print(f"Mean result shape: {mean_result.shape}")
    print(f"Expected shape: (10,) - Match: {mean_result.shape == (10,)}")
    
    print("\n=== Testing Index Selection ===")
    data_array = np.array([1.0, 2.0, 3.0, 4.0, 5.0], dtype=np.float32)
    indices = np.array([0, 2, 4], dtype=np.int32)
    
    selected = index_select(data_array, indices)
    print(f"Original: {data_array}")
    print(f"Indices: {indices}")
    print(f"Selected: {selected}")
    print(f"Expected: [1.0, 3.0, 5.0] - Match: {np.allclose(selected, [1.0, 3.0, 5.0])}")
    
    print("Example 1: Basic type annotations test passed")
    
except Exception as e:
    print(f"Example 1: Error in basic annotations test: {e}")
    import traceback
    traceback.print_exc()
    raise  # re-raise: a broken package must fail this example, not print past it

## Example 2: Runtime Type Checking and Validation

Demonstrate runtime type checking using jaxtyping decorators and validation.

In [ ]:
# Example 2: Runtime type checking and validation
from jaxtyping import Float, Int, jaxtyped
import jaxtyping
import numpy as np
# A bare `@jaxtyped` only opens a dimension-binding context and checks NOTHING (jaxtyping >= 0.2.24 warns
# about it); the annotations are enforced only when a typechecker is passed, as jaxtyping's docs require.
from beartype import beartype

try:
    print("=== Runtime Type Checking Setup ===")
    
    # Enable runtime type checking with decorator
    @jaxtyped(typechecker=beartype)
    def validated_linear_transform(
        weights: Float[np.ndarray, "out_dim in_dim"],
        inputs: Float[np.ndarray, "batch in_dim"],
        bias: Float[np.ndarray, "out_dim"]
    ) -> Float[np.ndarray, "batch out_dim"]:
        """Linear transformation with runtime validation"""
        return inputs @ weights.T + bias
    
    @jaxtyped(typechecker=beartype)
    def validated_convolution_1d(
        signal: Float[np.ndarray, "length"],
        kernel: Float[np.ndarray, "kernel_size"]
    ) -> Float[np.ndarray, "output_length"]:
        """1D convolution with type checking"""
        return np.convolve(signal, kernel, mode='valid')
    
    print("Functions with runtime type checking defined")
    
    print("\n=== Valid Input Testing ===")
    # Test with valid inputs
    weights = np.random.randn(5, 3).astype(np.float32)  # 5 output, 3 input
    inputs = np.random.randn(10, 3).astype(np.float32)   # batch=10, input=3
    bias = np.random.randn(5).astype(np.float32)         # 5 output
    
    print(f"Weights shape: {weights.shape}")
    print(f"Inputs shape: {inputs.shape}")
    print(f"Bias shape: {bias.shape}")
    
    try:
        result = validated_linear_transform(weights, inputs, bias)
        print(f"Valid call succeeded - Result shape: {result.shape}")
        print(f"Expected shape: (10, 5) - Match: {result.shape == (10, 5)}")
    except Exception as e:
        print(f"Valid call failed: {e}")
        raise
    
    print("\n=== Convolution Testing ===")
    signal = np.array([1.0, 2.0, 3.0, 4.0, 5.0], dtype=np.float32)
    kernel = np.array([0.5, -0.5], dtype=np.float32)
    
    print(f"Signal shape: {signal.shape}")
    print(f"Kernel shape: {kernel.shape}")
    
    try:
        conv_result = validated_convolution_1d(signal, kernel)
        print(f"Convolution result: {conv_result}")
        print(f"Result shape: {conv_result.shape}")
    except Exception as e:
        print(f"Convolution failed: {e}")
        raise
    
    print("\n=== Type Error Testing ===")
    # Test with invalid types/shapes
    invalid_tests = [
        ("Wrong weight shape", lambda: validated_linear_transform(
            np.random.randn(3, 5).astype(np.float32),  # Should be (5, 3)
            inputs, bias)),
        ("Wrong input dtype", lambda: validated_linear_transform(
            weights, inputs.astype(np.int32), bias)),  # Should be float
        ("Mismatched dimensions", lambda: validated_linear_transform(
            weights, np.random.randn(10, 4).astype(np.float32), bias))  # Should be (10, 3)
    ]
    
    for test_name, test_func in invalid_tests:
        try:
            test_func()
        except jaxtyping.TypeCheckError as e:
            print(f"{test_name}: Caught error as expected - {type(e).__name__}")
        else:
            raise AssertionError(f"{test_name}: jaxtyping accepted a call its annotations forbid")
    
    print("\n=== Shape Validation Details ===")
    # Demonstrate detailed shape checking
    def analyze_array_compliance(arr, expected_annotation):
        """Analyze if array matches jaxtyping annotation"""
        print(f"Array shape: {arr.shape}")
        print(f"Array dtype: {arr.dtype}")
        print(f"Expected: {expected_annotation}")
        
        # Basic compliance checks
        is_float = arr.dtype.kind == 'f'
        is_2d = len(arr.shape) == 2
        
        print(f"Is float type: {is_float}")
        print(f"Is 2D: {is_2d}")
    
    test_array = np.random.randn(8, 12).astype(np.float32)
    analyze_array_compliance(test_array, "Float[np.ndarray, 'batch features']")
    
    print("Example 2: Runtime type checking test passed")
    
except Exception as e:
    print(f"Example 2: Error in type checking test: {e}")
    import traceback
    traceback.print_exc()
    raise  # re-raise: a broken package must fail this example, not print past it

## Example 3: Complex Shape Patterns and Named Dimensions

Demonstrate advanced shape patterns, broadcasting, and named dimension handling.

In [ ]:
# Example 3: Complex shape patterns and named dimensions
from jaxtyping import Float, Int, Bool
import numpy as np
from typing import Union

try:
    print("=== Named Dimension Patterns ===")
    
    def attention_mechanism(
        query: Float[np.ndarray, "batch seq_len d_model"],
        key: Float[np.ndarray, "batch key_len d_model"],
        value: Float[np.ndarray, "batch key_len d_model"]
    ) -> Float[np.ndarray, "batch seq_len d_model"]:
        """Simplified attention with named dimensions"""
        # Compute attention scores
        scores = np.matmul(query, key.transpose(0, 2, 1))  # (batch, seq_len, key_len)
        scores = scores / np.sqrt(query.shape[-1])  # Scale by sqrt(d_model)
        
        # Apply softmax
        attention_weights = np.exp(scores) / np.sum(np.exp(scores), axis=-1, keepdims=True)
        
        # Apply attention to values
        output = np.matmul(attention_weights, value)  # (batch, seq_len, d_model)
        return output
    
    def batch_norm_2d(
        input_tensor: Float[np.ndarray, "batch channels height width"]
    ) -> Float[np.ndarray, "batch channels height width"]:
        """2D batch normalization with shape annotations"""
        # Compute statistics across batch and spatial dimensions
        mean = np.mean(input_tensor, axis=(0, 2, 3), keepdims=True)
        var = np.var(input_tensor, axis=(0, 2, 3), keepdims=True)
        
        # Normalize
        normalized = (input_tensor - mean) / np.sqrt(var + 1e-8)
        return normalized
    
    print("Functions with complex shape patterns defined")
    
    print("\n=== Testing Attention Mechanism ===")
    batch_size, seq_len, key_len, d_model = 2, 8, 10, 64
    
    query = np.random.randn(batch_size, seq_len, d_model).astype(np.float32)
    key = np.random.randn(batch_size, key_len, d_model).astype(np.float32)
    value = np.random.randn(batch_size, key_len, d_model).astype(np.float32)
    
    print(f"Query shape: {query.shape} (batch={batch_size}, seq_len={seq_len}, d_model={d_model})")
    print(f"Key shape: {key.shape} (batch={batch_size}, key_len={key_len}, d_model={d_model})")
    print(f"Value shape: {value.shape}")
    
    attention_output = attention_mechanism(query, key, value)
    print(f"Attention output shape: {attention_output.shape}")
    print(f"Expected: ({batch_size}, {seq_len}, {d_model}) - Match: {attention_output.shape == (batch_size, seq_len, d_model)}")
    
    print("\n=== Testing 2D Batch Normalization ===")
    batch, channels, height, width = 4, 16, 32, 32
    input_4d = np.random.randn(batch, channels, height, width).astype(np.float32)
    
    print(f"Input shape: {input_4d.shape} (NCHW format)")
    
    norm_output = batch_norm_2d(input_4d)
    print(f"Normalized output shape: {norm_output.shape}")
    
    # Verify normalization properties
    channel_means = np.mean(norm_output, axis=(0, 2, 3))
    channel_vars = np.var(norm_output, axis=(0, 2, 3))
    
    print(f"Channel means (should be ~0): max={np.max(np.abs(channel_means)):.6f}")
    print(f"Channel variances (should be ~1): mean={np.mean(channel_vars):.6f}")
    
    print("\n=== Broadcasting Patterns ===")
    
    def elementwise_operation(
        tensor_a: Float[np.ndarray, "batch height width"],
        tensor_b: Float[np.ndarray, "batch 1 1"]  # Broadcasts across height/width
    ) -> Float[np.ndarray, "batch height width"]:
        """Element-wise operation with broadcasting"""
        return tensor_a + tensor_b
    
    def weighted_sum(
        features: Float[np.ndarray, "batch n_features"],
        weights: Float[np.ndarray, "n_features"]  # Broadcasts across batch
    ) -> Float[np.ndarray, "batch"]:
        """Weighted sum with broadcasting"""
        return np.sum(features * weights[np.newaxis, :], axis=1)
    
    # Test broadcasting
    tensor_a = np.random.randn(3, 64, 64).astype(np.float32)
    tensor_b = np.random.randn(3, 1, 1).astype(np.float32)
    
    broadcast_result = elementwise_operation(tensor_a, tensor_b)
    print(f"Broadcasting result shape: {broadcast_result.shape}")
    print(f"Original A shape: {tensor_a.shape}, B shape: {tensor_b.shape}")
    
    # Test weighted sum
    features = np.random.randn(5, 20).astype(np.float32)
    weights = np.random.randn(20).astype(np.float32)
    
    weighted_result = weighted_sum(features, weights)
    print(f"Weighted sum result shape: {weighted_result.shape}")
    print(f"Features shape: {features.shape}, Weights shape: {weights.shape}")
    
    print("Example 3: Complex shape patterns test passed")
    
except Exception as e:
    print(f"Example 3: Error in complex patterns test: {e}")
    import traceback
    traceback.print_exc()
    raise  # re-raise: a broken package must fail this example, not print past it

## Example 4: Generic Type Annotations and Tree Structures

Demonstrates framework-agnostic type annotations and nested tree structure handling without JAX dependencies.

In [ ]:
# Example 4: Generic type annotations and tree structures
from jaxtyping import Float, Int
import numpy as np
from typing import Dict, List, Any, Union

try:
    print("=== Generic Array Type Annotations ===")
    
    # Use Float[np.ndarray, ...] for type annotations
    def generic_normalize(
        data: Float[np.ndarray, "batch features"]
    ) -> Float[np.ndarray, "batch features"]:
        """Normalize data across features"""
        mean = np.mean(data, axis=0, keepdims=True)
        std = np.std(data, axis=0, keepdims=True)
        return (data - mean) / (std + 1e-8)
    
    def cross_framework_dot_product(
        a: Float[np.ndarray, "n"],
        b: Float[np.ndarray, "n"]
    ) -> float:  # Scalar result
        """Dot product returning scalar"""
        return float(np.dot(a, b))
    
    def matrix_chain_multiply(
        matrices: List[Float[np.ndarray, "n n"]]
    ) -> Float[np.ndarray, "n n"]:
        """Chain multiply a list of matrices"""
        result = matrices[0]
        for mat in matrices[1:]:
            result = result @ mat
        return result
    
    print("Generic array functions defined")
    
    print("\n=== Testing with NumPy Arrays ===")
    np_data = np.random.randn(10, 5).astype(np.float32)
    np_vector_a = np.array([1.0, 2.0, 3.0], dtype=np.float32)
    np_vector_b = np.array([4.0, 5.0, 6.0], dtype=np.float32)
    
    np_normalized = generic_normalize(np_data)
    np_dot = cross_framework_dot_product(np_vector_a, np_vector_b)
    
    print(f"NumPy normalized shape: {np_normalized.shape}")
    print(f"Normalized mean per feature: {np.mean(np_normalized, axis=0)}")
    print(f"Normalized std per feature: {np.std(np_normalized, axis=0)}")
    print(f"NumPy dot product: {np_dot} (expected: 32.0)")
    
    # Test matrix chain multiplication
    matrices = [np.eye(4, dtype=np.float32) * (i + 1) for i in range(3)]
    chain_result = matrix_chain_multiply(matrices)
    print(f"\nMatrix chain result diagonal: {np.diag(chain_result)}")
    
    # Test with different dtypes
    int_array = np.array([1, 2, 3, 4, 5], dtype=np.int32)
    float_array = np.array([1.0, 2.0, 3.0, 4.0, 5.0], dtype=np.float64)
    
    print(f"\nArray dtype variations:")
    print(f"Integer array dtype: {int_array.dtype}")
    print(f"Float array dtype: {float_array.dtype}")
    
    print("\n=== Nested Structure Annotations (Tree-like) ===")
    
    # Define tree structures with type annotations (without PyTree import)
    ModelState = Dict[str, Union[Dict, np.ndarray, float, int]]
    
    def process_model_state(
        state: ModelState
    ) -> ModelState:
        """Process a model state tree structure"""
        # Simulate processing each leaf in the tree
        def process_leaf(leaf):
            if hasattr(leaf, 'shape'):  # It's an array
                return leaf * 0.99  # Apply decay
            else:
                return leaf  # Keep non-arrays as-is
        
        # Simple tree map simulation
        if isinstance(state, dict):
            return {k: process_leaf(v) if not isinstance(v, dict) else process_model_state(v) 
                   for k, v in state.items()}
        elif isinstance(state, (list, tuple)):
            return type(state)(process_leaf(item) if not isinstance(item, (dict, list, tuple)) 
                             else process_model_state(item) for item in state)
        else:
            return process_leaf(state)
    
    # Create nested model state structure
    model_state: ModelState = {
        'layers': {
            'dense1': {
                'weights': np.random.randn(128, 64).astype(np.float32),
                'bias': np.random.randn(128).astype(np.float32)
            },
            'dense2': {
                'weights': np.random.randn(10, 128).astype(np.float32),
                'bias': np.random.randn(10).astype(np.float32)
            }
        },
        'optimizer': {
            'momentum': np.random.randn(128, 64).astype(np.float32),
            'learning_rate': 0.001
        },
        'epoch': 42
    }
    
    print("Model state structure:")
    print(f"Dense1 weights shape: {model_state['layers']['dense1']['weights'].shape}")
    print(f"Dense2 weights shape: {model_state['layers']['dense2']['weights'].shape}")
    print(f"Learning rate: {model_state['optimizer']['learning_rate']}")
    print(f"Current epoch: {model_state['epoch']}")
    
    # Process the tree
    updated_state = process_model_state(model_state)
    
    # Verify processing
    original_weight = model_state['layers']['dense1']['weights'][0, 0]
    updated_weight = updated_state['layers']['dense1']['weights'][0, 0]
    
    print(f"\nWeight update: {original_weight:.6f} -> {updated_weight:.6f}")
    print(f"Decay factor: {updated_weight / original_weight:.6f} (expected: 0.99)")
    print(f"Non-array preserved: epoch {updated_state['epoch']} (unchanged)")
    
    print("\n=== Complex Tree Operations ===")
    
    def tree_statistics(
        tree: ModelState
    ) -> Dict[str, Any]:
        """Compute statistics over arrays in a tree"""
        stats = {'total_params': 0, 'mean_magnitude': [], 'shapes': []}
        
        def collect_stats(obj, path=""):
            if isinstance(obj, dict):
                for k, v in obj.items():
                    collect_stats(v, f"{path}.{k}" if path else k)
            elif isinstance(obj, (list, tuple)):
                for i, v in enumerate(obj):
                    collect_stats(v, f"{path}[{i}]")
            elif hasattr(obj, 'shape'):  # Array-like
                stats['total_params'] += int(obj.size)
                stats['mean_magnitude'].append(float(np.mean(np.abs(obj))))
                stats['shapes'].append((path, tuple(obj.shape)))
        
        collect_stats(tree)
        return stats
    
    # Compute statistics
    tree_stats = tree_statistics(model_state)
    
    print(f"Tree statistics:")
    print(f"Total parameters: {tree_stats['total_params']:,}")
    print(f"Average magnitude: {np.mean(tree_stats['mean_magnitude']):.6f}")
    print(f"Number of arrays: {len(tree_stats['shapes'])}")
    
    # Show array locations and shapes
    print("Array shapes in tree:")
    for path, shape in tree_stats['shapes'][:4]:  # Show first 4
        print(f"  {path}: {shape}")
    
    print("\n=== Nested Tree Transformations ===")
    
    def scale_tree(
        tree: ModelState,
        scale: float
    ) -> ModelState:
        """Scale all arrays in tree by a constant"""
        def scale_leaf(leaf):
            if hasattr(leaf, 'shape'):
                return leaf * scale
            return leaf
        
        if isinstance(tree, dict):
            return {k: scale_leaf(v) if not isinstance(v, (dict, list, tuple)) else scale_tree(v, scale)
                   for k, v in tree.items()}
        elif isinstance(tree, (list, tuple)):
            return type(tree)(scale_leaf(item) if not isinstance(item, (dict, list, tuple)) 
                            else scale_tree(item, scale) for item in tree)
        else:
            return scale_leaf(tree)
    
    # Test scaling
    scaled_state = scale_tree(model_state, 2.0)
    
    orig_weight_val = model_state['layers']['dense1']['weights'][0, 0]
    scaled_weight_val = scaled_state['layers']['dense1']['weights'][0, 0]
    
    print(f"Scaling transformation:")
    print(f"Original: {orig_weight_val:.6f}")
    print(f"Scaled (2x): {scaled_weight_val:.6f}")
    print(f"Ratio: {scaled_weight_val / orig_weight_val:.1f}")
    print(f"Non-array unchanged: learning_rate {scaled_state['optimizer']['learning_rate']}")
    
    print("\n=== Batch Operations with Type Safety ===")
    
    def batch_process(
        inputs: Float[np.ndarray, "batch input_dim"],
        weights: Float[np.ndarray, "output_dim input_dim"]
    ) -> Float[np.ndarray, "batch output_dim"]:
        """Batch matrix multiplication with type annotations"""
        return inputs @ weights.T
    
    # Test batch processing
    batch_inputs = np.random.randn(32, 20).astype(np.float32)
    batch_weights = np.random.randn(10, 20).astype(np.float32)
    
    batch_output = batch_process(batch_inputs, batch_weights)
    print(f"Batch input shape: {batch_inputs.shape}")
    print(f"Weights shape: {batch_weights.shape}")
    print(f"Batch output shape: {batch_output.shape}")
    print(f"Expected: (32, 10) - Match: {batch_output.shape == (32, 10)}")
    
    print("Example 4: Generic types and tree structures test passed")
    
except Exception as e:
    print(f"Example 4: Error in generic types test: {e}")
    import traceback
    traceback.print_exc()
    raise  # re-raise: a broken package must fail this example, not print past it

## Example 5: Production Type Checking and Integration Patterns

Demonstrate production-ready type checking patterns and integration with other tools.

In [ ]:
# Example 5: Production type checking and integration patterns
from jaxtyping import Float, Int, Bool, jaxtyped
import numpy as np
from typing import Protocol, Union, Optional, Tuple
import functools

try:
    print("=== Production Type Checking Patterns ===")
    
    # Production-ready linear layer with comprehensive type checking
    @jaxtyped
    def production_linear_layer(
        inputs: Float[np.ndarray, "batch input_dim"],
        weights: Float[np.ndarray, "output_dim input_dim"], 
        bias: Optional[Float[np.ndarray, "output_dim"]] = None,
        activation: str = "relu"
    ) -> Float[np.ndarray, "batch output_dim"]:
        """Production linear layer with full type checking"""
        # Linear transformation
        output = inputs @ weights.T
        
        # Add bias if provided
        if bias is not None:
            output = output + bias
        
        # Apply activation
        if activation == "relu":
            output = np.maximum(0, output)
        elif activation == "sigmoid":
            output = 1 / (1 + np.exp(-np.clip(output, -500, 500)))  # Numerical stability
        elif activation == "tanh":
            output = np.tanh(output)
        
        return output
    
    print("Production linear layer with type checking defined")
    
    print("\n=== Comprehensive Model Class ===")
    
    class TypedMLPModel:
        """MLP model with comprehensive jaxtyping annotations"""
        
        def __init__(
            self,
            layer1_weights: Float[np.ndarray, "hidden1 input_dim"],
            layer1_bias: Float[np.ndarray, "hidden1"],
            layer2_weights: Float[np.ndarray, "hidden2 hidden1"],
            layer2_bias: Float[np.ndarray, "hidden2"],
            output_weights: Float[np.ndarray, "output_dim hidden2"],
            output_bias: Float[np.ndarray, "output_dim"]
        ):
            self.layer1_weights = layer1_weights
            self.layer1_bias = layer1_bias
            self.layer2_weights = layer2_weights
            self.layer2_bias = layer2_bias
            self.output_weights = output_weights
            self.output_bias = output_bias
        
        @jaxtyped
        def forward(
            self, 
            x: Float[np.ndarray, "batch input_dim"]
        ) -> Float[np.ndarray, "batch output_dim"]:
            """Forward pass with type checking"""
            # Layer 1
            h1 = production_linear_layer(x, self.layer1_weights, self.layer1_bias, "relu")
            
            # Layer 2  
            h2 = production_linear_layer(h1, self.layer2_weights, self.layer2_bias, "relu")
            
            # Output layer
            output = production_linear_layer(h2, self.output_weights, self.output_bias, "sigmoid")
            
            return output
        
        @jaxtyped
        def predict_proba(
            self,
            x: Float[np.ndarray, "batch input_dim"]
        ) -> Float[np.ndarray, "batch output_dim"]:
            """Get prediction probabilities"""
            return self.forward(x)
        
        @jaxtyped 
        def predict(
            self,
            x: Float[np.ndarray, "batch input_dim"],
            threshold: float = 0.5
        ) -> Bool[np.ndarray, "batch output_dim"]:
            """Get binary predictions"""
            probs = self.forward(x)
            return probs > threshold
    
    # Create model with proper dimensions
    input_dim, hidden1, hidden2, output_dim = 10, 64, 32, 3
    batch_size = 8
    
    model = TypedMLPModel(
        layer1_weights=np.random.randn(hidden1, input_dim).astype(np.float32),
        layer1_bias=np.random.randn(hidden1).astype(np.float32),
        layer2_weights=np.random.randn(hidden2, hidden1).astype(np.float32),
        layer2_bias=np.random.randn(hidden2).astype(np.float32),
        output_weights=np.random.randn(output_dim, hidden2).astype(np.float32),
        output_bias=np.random.randn(output_dim).astype(np.float32)
    )
    
    print(f"Model created with architecture: {input_dim} -> {hidden1} -> {hidden2} -> {output_dim}")
    
    print("\n=== Model Testing ===")
    # Test model with valid inputs
    test_input = np.random.randn(batch_size, input_dim).astype(np.float32)
    
    # Forward pass
    probabilities = model.predict_proba(test_input)
    predictions = model.predict(test_input)
    
    print(f"Input shape: {test_input.shape}")
    print(f"Probabilities shape: {probabilities.shape}")
    print(f"Predictions shape: {predictions.shape}")
    print(f"Probability range: [{np.min(probabilities):.3f}, {np.max(probabilities):.3f}]")
    print(f"Predictions dtype: {predictions.dtype}")
    
    print("\n=== Error Handling and Validation ===")
    
    # Custom validation decorator
    def validate_array_properties(func):
        """Decorator for additional array validation"""
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            # Check for NaN/Inf in array arguments
            for arg in args:
                if hasattr(arg, 'shape') and np.any(~np.isfinite(arg)):
                    raise ValueError(f"Array contains NaN/Inf values in {func.__name__}")
            
            result = func(*args, **kwargs)
            
            # Check result for NaN/Inf
            if hasattr(result, 'shape') and np.any(~np.isfinite(result)):
                raise ValueError(f"Function {func.__name__} produced NaN/Inf values")
            
            return result
        return wrapper
    
    @validate_array_properties
    @jaxtyped
    def robust_computation(
        data: Float[np.ndarray, "n features"],
        weights: Float[np.ndarray, "features"]
    ) -> Float[np.ndarray, "n"]:
        """Computation with additional validation"""
        return np.sum(data * weights, axis=1)
    
    # Test robust computation
    valid_data = np.random.randn(5, 8).astype(np.float32)
    valid_weights = np.random.randn(8).astype(np.float32)
    
    try:
        result = robust_computation(valid_data, valid_weights)
        print(f"Robust computation succeeded: shape {result.shape}")
    except Exception as e:
        print(f"Robust computation failed: {e}")
        raise
    
    # Test with invalid data (containing NaN)
    invalid_data = valid_data.copy()
    invalid_data[0, 0] = np.nan
    
    try:
        result = robust_computation(invalid_data, valid_weights)
    except ValueError as e:
        assert "NaN/Inf" in str(e), f"rejected for an unexpected reason: {e}"
        print(f"Robust computation correctly caught NaN: {str(e)[:50]}...")
    else:
        raise AssertionError(f"robust_computation accepted a NaN input and returned {result!r}")
    
    print("\n=== Type Safety Summary ===")
    
    # Summary of type safety features used
    features_used = [
        "Shape annotations with named dimensions",
        "Multi-dimensional array type checking", 
        "Optional parameter type annotations",
        "Boolean array return types",
        "Custom validation decorators",
        "Runtime type checking with @jaxtyped",
        "Production-ready model class design"
    ]
    
    print("Type safety features demonstrated:")
    for i, feature in enumerate(features_used, 1):
        print(f"{i}. {feature}")
    
    total_params = sum(w.size for w in [
        model.layer1_weights, model.layer1_bias, 
        model.layer2_weights, model.layer2_bias, 
        model.output_weights, model.output_bias
    ])
    print(f"\nTotal parameters in model: {total_params:,}")
    
    print("\n=== Integration Pattern Examples ===")
    
    # Demonstrate integration with existing code
    def legacy_function(data):
        """Legacy function without type annotations"""
        return data * 2
    
    @jaxtyped
    def typed_wrapper(
        data: Float[np.ndarray, "n features"]
    ) -> Float[np.ndarray, "n features"]:
        """Type-safe wrapper around legacy code"""
        return legacy_function(data)
    
    # Test integration
    test_data = np.random.randn(5, 3).astype(np.float32)
    wrapped_result = typed_wrapper(test_data)
    
    print(f"Legacy function integration:")
    print(f"Input shape: {test_data.shape}")
    print(f"Output shape: {wrapped_result.shape}")
    print(f"Type safety maintained: {wrapped_result.shape == test_data.shape}")
    
    print("Example 5: Production type checking patterns test passed")
    
except Exception as e:
    print(f"Example 5: Error in production patterns test: {e}")
    import traceback
    traceback.print_exc()
    raise  # re-raise: a broken package must fail this example, not print past it